# Phase 2b · Screen sampled pathogen–host evidence

Screen a reproducible sample of PubMed records for each configured pathogen–host pair. A pair is accepted as soon as one completed sample assessment answers yes with a supporting span quoted exactly from that article’s abstract. A title alone cannot establish acceptance. Records without abstracts remain unassessed and keep the pair incomplete unless another abstract supports it. The sample size is configured in the first code cell of `00_workflow`; increasing the cap reuses previous pair searches and abstracts. The handler is provider-agnostic and no model calls are made unless the run switch is enabled.

## Step 1 · Load the selected run, pair cache, and sampling policy

In [ ]:
import hashlib, json, os, sys
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import display
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository.')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id
from pathogen_atlas_v4.runs import resolve_run
from pathogen_atlas_v4.pubmed_client import prepare_pair_screen_samples
from pathogen_atlas_v4.pathogen_screening import build_pathogen_screen_tasks, select_pathogen_host_pairs
from pathogen_atlas_v4.decision_runner import run_decisions
CONFIG = yaml.safe_load((PROJECT_ROOT / 'configs/pilot_v1.yaml').read_text(encoding='utf-8'))
RUN_ID = configured_run_id(PROJECT_ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('02b_pathogen_screening', RUN_ID)
TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
PATHOGENS = pd.read_parquet(TAXONOMY_ROOT / 'pathogens.parquet')
HOSTS = pd.read_parquet(TAXONOMY_ROOT / 'hosts.parquet')
ALIASES = pd.read_parquet(TAXONOMY_ROOT / 'pathogen_aliases.parquet', columns=['pathogen_id','name'])
host_groups = {k: g.host_id.astype(str).tolist() for k,g in HOSTS.groupby('host_group')}
pathogen_groups = {k: g.pathogen_id.astype(str).tolist() for k,g in PATHOGENS.groupby('pathogen_group')}
resolved = resolve_run(CONFIG, known_host_ids=set(HOSTS.host_id.astype(str)), known_pathogen_ids=set(PATHOGENS.pathogen_id.astype(str)), host_group_ids=host_groups, pathogen_group_ids=pathogen_groups)
SAMPLING = CONFIG['acquisition'].get('sampling', {})
SAMPLE_MAXIMUM = SAMPLING.get('max_pmids_per_pair', 200)
SAMPLE_OLDEST = int(SAMPLING.get('oldest_pmids_per_pair', 30))
SAMPLE_SEED = int(SAMPLING.get('random_seed', 42))
SAMPLE_POLICY = str(SAMPLING.get('sample_policy_version', 'stable-prefix-hash-v1'))
cache_identity = {'query_version': CONFIG['acquisition']['query_strategy_version'], 'pathogen_literature_query_version': CONFIG['acquisition']['pathogen_literature_query_version'], 'taxonomy_snapshot': TAXONOMY_ROOT.parent.name}
cache_key = hashlib.sha256(json.dumps(cache_identity, sort_keys=True, default=list).encode()).hexdigest()
CACHE_DIR = ROOT / 'data/v4/cache/literature' / cache_key
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
PROMPTS = yaml.safe_load((ROOT / CONFIG['decisions']['prompt_catalogue']).read_text(encoding='utf-8'))
SAMPLE_MATCHES_PATH = CACHE_DIR / 'tables/pathogen_screen_matches.parquet'
ARTICLES_PATH = CACHE_DIR / 'tables/articles.parquet'
SHARED_DECISION_CACHE = ROOT / 'data/v4/cache/decisions'
SCREEN_LEDGER_PATH = SHARED_DECISION_CACHE / 'pathogen_screen_ledger.parquet'
PAIR_SELECTION_PATH = RUN_DIR / 'pathogen_host_selection.parquet'
print('Run:', RUN_ID, '| candidate pairs:', len(resolved.pairs), '| sample cap:', SAMPLE_MAXIMUM, '| oldest PMID IDs:', SAMPLE_OLDEST, '| seed:', SAMPLE_SEED)
print('Pair cache:', CACHE_DIR, '| log:', LOG_PATH)
LOG.info('Pathogen screen setup: pairs=%d cap=%s oldest=%d seed=%d cache=%s', len(resolved.pairs), SAMPLE_MAXIMUM, SAMPLE_OLDEST, SAMPLE_SEED, CACHE_DIR)

## Step 2 · Select stable sample PMIDs and retrieve their abstracts

In [ ]:
RUN_SAMPLE_PREPARATION = False  # Set True after Phase 2 pair searches have completed.
if RUN_SAMPLE_PREPARATION:
    NCBI_EMAIL = os.environ.get('NCBI_EMAIL') or input('NCBI contact email: ').strip()
    NCBI_API_KEY = os.environ.get('NCBI_API_KEY') or None
    sample_summary = prepare_pair_screen_samples(
        resolved.pairs, CACHE_DIR, sample_maximum=SAMPLE_MAXIMUM, oldest_count=SAMPLE_OLDEST,
        seed=SAMPLE_SEED, sample_policy_version=SAMPLE_POLICY, email=NCBI_EMAIL,
        api_key=NCBI_API_KEY, logger=LOG)
    display(sample_summary)
else:
    print('Sample preparation skipped. Set RUN_SAMPLE_PREPARATION=True to prepare the pair samples.')
    print('Existing sample table:', SAMPLE_MATCHES_PATH.exists())
if SAMPLE_MATCHES_PATH.exists():
    sample_matches = pd.read_parquet(SAMPLE_MATCHES_PATH)
    articles = pd.read_parquet(ARTICLES_PATH)
    print('Sampled pair records:', f'{len(sample_matches):,}', '| abstracts available:', f'{int(sample_matches.abstract_available.sum()):,}')
else:
    sample_matches = pd.DataFrame(columns=['pathogen_id','host_id','pmid','query_hash','query_version','pair_search_count','sample_selection'])
    articles = pd.DataFrame(columns=['pmid','title','abstract'])
LOG.info('Sample inputs loaded: rows=%d articles=%d', len(sample_matches), len(articles))

## Step 3 · Build and inspect pathogen yes/no tasks

In [ ]:
PROMPT_MODEL_VERSION = str(CONFIG['decisions'].get('model_version', 'configured-per-run'))
TASKS = build_pathogen_screen_tasks(sample_matches, articles, PATHOGENS, HOSTS, PROMPTS,
    run_id=RUN_ID, run_config_version=str(CONFIG['run']['config_version']), model_version=PROMPT_MODEL_VERSION)
print('Screen tasks:', f'{len(TASKS):,}')
if TASKS:
    print(json.dumps(TASKS[0]['prompt'], indent=2, ensure_ascii=False))
LOG.info('Built pathogen screen tasks=%d', len(TASKS))

## Step 4 · Run or resume the screening ledger

In [ ]:
PATHOGEN_SCREEN_HANDLER = None  # Configure a callable(task) -> {answer, confidence, evidence_span, rationale}.
RUN_SCREENING = False
MAX_ATTEMPTS = int(CONFIG.get('decisions', {}).get('max_attempts', 2))
if RUN_SCREENING:
    if not TASKS or not callable(PATHOGEN_SCREEN_HANDLER):
        raise RuntimeError('Prepare sampled article tasks and configure a provider handler before screening.')
    LEDGER = run_decisions(TASKS, PATHOGEN_SCREEN_HANDLER, ledger_path=SCREEN_LEDGER_PATH,
                           raw_response_path=SHARED_DECISION_CACHE / 'pathogen_screen_raw_responses.jsonl',
                           max_attempts=MAX_ATTEMPTS)
    display(LEDGER.groupby(['status','answer'], dropna=False).size().rename('records').to_frame())
else:
    LEDGER = pd.read_parquet(SCREEN_LEDGER_PATH) if SCREEN_LEDGER_PATH.exists() else pd.DataFrame()
    print('Provider calls skipped. Set RUN_SCREENING=True after configuring PATHOGEN_SCREEN_HANDLER.')
print('Ledger:', SCREEN_LEDGER_PATH, '| rows:', len(LEDGER))
LOG.info('Pathogen screen ledger rows=%d', len(LEDGER))

## Step 5 · Apply the acceptance rule for each host

In [ ]:
if not sample_matches.empty:
    pair_selection = select_pathogen_host_pairs(resolved.pairs, sample_matches, LEDGER)
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    pair_selection.to_parquet(PAIR_SELECTION_PATH, index=False)
    display(pair_selection.selection_status.value_counts(dropna=False).rename('pairs').to_frame())
    display(pair_selection.loc[pair_selection.selection_status.eq('accepted')].head(20))
    print('Accepted pathogen–host pairs:', int(pair_selection.selected_as_pathogen_for_host.sum()))
else:
    pair_selection = pd.DataFrame()
    pair_selection = select_pathogen_host_pairs(resolved.pairs, sample_matches, LEDGER)
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    pair_selection.to_parquet(PAIR_SELECTION_PATH, index=False)
    print('No abstract sample is available yet; candidate pairs are recorded as incomplete or without sampled records.')
LOG.info('Pair selection saved=%s accepted=%d', not pair_selection.empty, int(pair_selection.selected_as_pathogen_for_host.sum()) if not pair_selection.empty else 0)

Acceptance is pair-specific: one completed yes response with an exact supporting span accepts that pathogen for that host. A pair with no yes is rejected only after every sampled PMID has a completed screen; otherwise it remains incomplete. No confidence threshold is applied to the yes/no acceptance rule.